# # Install dependencies

In [ ]:
!pip install -q --user scikit-learn torch pytorch-lightning


# # Load training data

In [ ]:
"""Load the training data.

Pick the form that matches your storage:

    df = data.connection(conn_name).load("SELECT * FROM iris")   # SQL Explorer connection
    df = data.load("/home/jovyan/data/iris.csv")                 # local file (csv/parquet)
    df = data.load("s3://my-bucket/iris.parquet")                # S3 object

`data.split` shuffles rows by default (use `shuffle=False` to keep order,
e.g. for time-series). `random_state` makes the split reproducible.

`prepare()` is demo scaffolding — replace with your own data source in production.
"""
from src.demo import prepare
conn_name = prepare()

from nubison_model import data

df = data.connection(conn_name).load("SELECT * FROM iris")
datasets = data.split(df, {"train": 0.6, "val": 0.2, "test": 0.2}, random_state=42)

for name, sub in datasets.items():
    print(f"{name:6s} rows={len(sub):3d}")


# # Train (PyTorch Lightning)

In [ ]:
"""Define the module in ``src/iris_lightning.py`` so ``infer_lightning.ipynb``
can unpickle it later. A class defined inside this notebook is pickled as
``__main__.IrisLightning`` and won't resolve in another process.
"""
import torch
from torch.utils.data import DataLoader, TensorDataset
import pytorch_lightning as pl
from src.iris_lightning import IrisLightning


In [ ]:
"""`train(datasets, target, *, ...)` parameters:
    datasets      — dict from `data.split` (must include "train";
                    "val" enables `t.X_val / t.y_val` + auto-scoring;
                    "test" populates `t.X_test / t.y_test`)
    target        — label column name(s); single string or list of strings.
    model_type    — "classifier" / "regressor" / free-form string tag.
    weights_path  — default "src/weights.pkl".

mlflow.pytorch.autolog hooks `LightningModule.save_hyperparameters()` and
`self.log()`. Passing `logger=False` to `pl.Trainer` disables both hooks, so
keep the default Lightning logger active for params/metrics to be captured.
"""
from nubison_model import train

EPOCHS, BATCH = 30, 16

with train(datasets=datasets, target=["target"], model_type="classifier") as t:
    X_train = torch.tensor(t.X_train.values, dtype=torch.float32)
    y_train = torch.tensor(t.y_train.values.ravel(), dtype=torch.long)
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH, shuffle=True)

    val_loader = None
    if t.X_val is not None and t.y_val is not None:
        X_val = torch.tensor(t.X_val.values, dtype=torch.float32)
        y_val = torch.tensor(t.y_val.values.ravel(), dtype=torch.long)
        val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=BATCH)

    model = IrisLightning(lr=0.01)
    trainer = pl.Trainer(
        max_epochs=EPOCHS,
        accelerator="cpu",
        enable_progress_bar=False,
        enable_checkpointing=False,
        # Do NOT pass `logger=False` — it disables the mlflow autolog hooks.
    )
    trainer.fit(model, train_loader, val_loader)

    t.save(model)

print(f"run_id: {t.run_id}")
